# Task 15: Distributed Data Parallel (DDP) Multi-GPU Node Sync Gateway with GLOO/NCCL

**Objective:** Master large-scale deep learning model partitioning and gradient synchronization across hardware nodes by deploying DDP routines.

This folder contains:
1. Multi-GPU training script (`ddp_train.py`)
2. Container environment (`Dockerfile`)
3. Orchestration gateway (`docker-compose.yml`)

The notebook contains structural walkthroughs and local execution templates.

In [ ]:
# Local multi-process DDP simulation hook using PyTorch multiprocessing
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn

def run_worker(rank, world_size):
    # Set master details
    import os
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    
    # Initialize process group using GLOO backend (fully supported on Windows/CPU!)
    dist.init_process_group("gloo", rank=rank, world_size=world_size)
    
    model = nn.Linear(10, 1).to(rank)
    # Wrap model in PyTorch DDP
    ddp_model = nn.parallel.DistributedDataParallel(model)
    
    # Generate dummy input w.r.t process rank
    inputs = torch.randn(4, 10)
    outputs = ddp_model(inputs)
    loss = outputs.sum()
    
    loss.backward()
    dist.destroy_process_group()
    print(f"Worker {rank} successfully executed training step!")

if __name__ == '__main__':
    world_size = 2
    # Spawn workers locally to simulate sync gateway
    mp.spawn(run_worker, args=(world_size,), nprocs=world_size, join=True)